# NEAT — NeuroEvolution of Augmenting Topologies

Implementation of the NEAT algorithm (Stanley & Miikkulainen, 2002) evaluated on three Gymnasium
benchmarks (CartPole, LunarLander, BipedalWalker), plus experiments on speciation, mutation rates and
network complexity.

Paper: https://nn.cs.utexas.edu/downloads/papers/stanley.ec02.pdf

Structure: implementation -> visualisation helpers -> 3 benchmarks -> 3 analyses -> genetic distance.
The notebook is written to run top-to-bottom (Restart & Run All).

In [ ]:
from __future__ import annotations

import copy
import itertools
from dataclasses import dataclass, replace
from enum import Enum
from typing import Callable
from collections import Counter, defaultdict

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym

## Genome configuration and activation functions

The original paper uses a steepened sigmoid `y = 1/(1+e^{-4.9x})` with range [0, 1]. That cannot express a
negative torque, so BipedalWalker (continuous actions in [-1, 1]) uses `tanh` instead.

In [ ]:
def neat_tanh(W, x):
    return np.tanh(W @ x)


def neat_sigmoid(W, x):
    # NEAT steepened sigmoid; the clip keeps exp() from overflowing for large weights
    return 1 / (1 + np.exp(np.clip(-4.9 * (W @ x), -500, 500)))


@dataclass
class GenomeConfig:
    cross_p_inherit_weaker: float = 0.5
    cross_p_stay_disabled: float = 0.75
    mut_p_add_node: float = 0.05
    mut_p_add_link: float = 0.1
    mut_p_replace: float = 0.1          # 1 - p_replace = p_perturbate
    mut_pertubation_range: float = 2.0
    mut_max_weight: float = 8.0
    mut_min_weight: float = -8.0
    mut_clip_weights: bool = True
    activation_fun: Callable = neat_tanh

## Genome (`Individual`)

A genome *is* a `networkx.DiGraph`: nodes carry a `type` (INPUT/OUTPUT/HIDDEN), edges carry
`innov`, `weight` and `enabled`. Aligning two genomes by innovation number is therefore a dictionary
join, and the acyclicity check is `nx.has_path`.

For the forward pass the graph is compiled once into a dense weight matrix `W`; because the graph is a
DAG of depth `d`, iterating `state <- act(W @ state)` exactly `d` times reproduces the true feed-forward
output.

In [ ]:
class NodeType(Enum):
    INPUT = 0
    OUTPUT = 1
    HIDDEN = 2


class Individual:
    """One genome / network / solution."""

    def __init__(self, config: GenomeConfig = None):
        self.config = config if config else GenomeConfig()
        self.network = nx.DiGraph()
        self.fitness = 0.0

    def add_node_gene(self, node_id: int, node_type: NodeType):
        self.network.add_node(node_id, type=node_type)

    def add_connection_gene(self, in_node: int, out_node: int, innov: int, weight: float, enabled: bool):
        self.network.add_edge(in_node, out_node, innov=innov, weight=weight, enabled=enabled)

    def n_hidden(self) -> int:
        return sum(1 for _, a in self.network.nodes(data=True) if a['type'] == NodeType.HIDDEN)

    def n_enabled_edges(self) -> int:
        return sum(1 for _, _, a in self.network.edges(data=True) if a['enabled'])

    def clone(self) -> Individual:
        child = Individual(self.config)
        child.network = self.network.copy()
        child.fitness = self.fitness      # FIX: fitness must survive cloning
        child.compile_network()
        return child

    def _get_innov_dict(self):
        """O(1) innovation lookups for crossover / distance."""
        return {attr['innov']: (u, v, attr) for u, v, attr in self.network.edges(data=True)}

    def distance(self, other: Individual, c1=1.0, c2=0.4) -> float:
        """Compatibility distance; excess and disjoint genes are merged into one term."""
        genes_self, genes_other = self._get_innov_dict(), other._get_innov_dict()
        innov_self, innov_other = set(genes_self), set(genes_other)

        matching = innov_self & innov_other
        non_homologous = len(innov_self ^ innov_other)

        if matching:
            w1 = np.array([genes_self[k][2]['weight'] for k in matching])
            w2 = np.array([genes_other[k][2]['weight'] for k in matching])
            weight_diff = float(np.mean(np.abs(w1 - w2)))
        else:
            weight_diff = 0.0

        N = max(len(innov_self), len(innov_other), 1)
        return (c1 * non_homologous / N) + (c2 * weight_diff)

    def crossover(self, other: Individual) -> Individual:
        """Innovation-number aligned crossover."""
        child = Individual(self.config)          # FIX: child used to get a DEFAULT config
        fitter, weaker = (self, other) if self.fitness >= other.fitness else (other, self)

        genes_fitter, genes_weaker = fitter._get_innov_dict(), weaker._get_innov_dict()
        child.network.add_nodes_from(fitter.network.nodes(data=True))

        # iterating only over the fitter parent's genes IS the rule
        # "excess and disjoint genes are inherited from the more fit parent"
        for innov_num, (u, v, attr) in genes_fitter.items():
            child_attr = copy.deepcopy(attr)
            if innov_num in genes_weaker:
                weaker_attr = genes_weaker[innov_num][2]
                if np.random.random() < self.config.cross_p_inherit_weaker:
                    child_attr['weight'] = weaker_attr['weight']
                if not attr['enabled'] or not weaker_attr['enabled']:
                    child_attr['enabled'] = np.random.random() > self.config.cross_p_stay_disabled
            child.network.add_edge(u, v, **child_attr)
        return child

    def mutate_weights(self):
        edges = list(self.network.edges(data=True))
        n = len(edges)
        replacements = np.random.uniform(self.config.mut_min_weight, self.config.mut_max_weight, n)
        perturbations = np.random.uniform(-self.config.mut_pertubation_range,
                                          self.config.mut_pertubation_range, n)
        action_probs = np.random.rand(n)

        for i, (_, _, attr) in enumerate(edges):
            if action_probs[i] < self.config.mut_p_replace:
                attr['weight'] = replacements[i]
            else:
                attr['weight'] += perturbations[i]
                if self.config.mut_clip_weights:
                    attr['weight'] = float(np.clip(attr['weight'],
                                                   self.config.mut_min_weight,
                                                   self.config.mut_max_weight))

    def mutate_genes(self, get_link_innovation: Callable, get_node_innovation: Callable):
        if np.random.random() < self.config.mut_p_add_link:
            nodes = list(self.network.nodes(data=True))
            in_node, in_attr = nodes[np.random.choice(len(nodes))]
            out_node, out_attr = nodes[np.random.choice(len(nodes))]

            valid_type = in_attr['type'] != NodeType.OUTPUT and out_attr['type'] != NodeType.INPUT
            is_new = not self.network.has_edge(in_node, out_node)
            not_self_loop = in_node != out_node
            # adding in->out closes a cycle iff a path out->in already exists
            no_cycle = not nx.has_path(self.network, out_node, in_node)

            if valid_type and not_self_loop and is_new and no_cycle:
                self.add_connection_gene(
                    in_node, out_node,
                    innov=get_link_innovation(in_node, out_node),
                    weight=float(np.random.uniform(self.config.mut_min_weight,
                                                   self.config.mut_max_weight)),
                    enabled=True)

        if np.random.random() < self.config.mut_p_add_node:
            # split N1 -> N2 into N1 -> Nnew -> N2
            enabled_edges = [(u, v, a) for u, v, a in self.network.edges(data=True) if a['enabled']]
            if not enabled_edges:
                return
            n_1, n_2, attr = enabled_edges[np.random.choice(len(enabled_edges))]
            attr['enabled'] = False

            new_node_id = get_node_innovation(attr['innov'])
            self.add_node_gene(new_node_id, NodeType.HIDDEN)
            # weight 1.0 then the original weight -> the new node is almost a no-op at birth,
            # so it does not wreck the genome's fitness immediately
            self.add_connection_gene(n_1, new_node_id,
                                     innov=get_link_innovation(n_1, new_node_id),
                                     weight=1.0, enabled=True)
            self.add_connection_gene(new_node_id, n_2,
                                     innov=get_link_innovation(new_node_id, n_2),
                                     weight=attr['weight'], enabled=True)

    def compile_network(self):
        """Compile the graph into a dense weight matrix for the forward pass."""
        self._node_idx_map = {key: i for i, key in enumerate(self.network.nodes())}
        n_nodes = len(self._node_idx_map)
        self._W = np.zeros((n_nodes, n_nodes))

        self._input_indices = [self._node_idx_map[n] for n, a in self.network.nodes(data=True)
                               if a['type'] == NodeType.INPUT]
        self._output_indices = [self._node_idx_map[n] for n, a in self.network.nodes(data=True)
                                if a['type'] == NodeType.OUTPUT]

        for u, v, attr in self.network.edges(data=True):
            if attr['enabled']:
                self._W[self._node_idx_map[v]][self._node_idx_map[u]] = attr['weight']

        self._network_depth = nx.dag_longest_path_length(self.network, weight=None) + 1

    def forward_pass(self, inputs: np.ndarray) -> np.ndarray:
        if len(inputs) != len(self._input_indices):
            raise ValueError("Number of input nodes doesn't match")

        state = np.zeros(len(self._node_idx_map))
        state[self._input_indices] = inputs        # FIX: was state[:len(inputs)], which silently
                                                   # assumed inputs occupy the first matrix indices
        for _ in range(self._network_depth):
            state = self.config.activation_fun(self._W, state)
            state[self._input_indices] = inputs    # re-clamp inputs each propagation step
        return state[self._output_indices]

## Population configuration

In [ ]:
@dataclass
class PopulationConfig:
    dropoff_age: int = 15               # a species that stops improving for this long is penalised
    p_selection: float = 0.2            # top 20 % of a species become parents
    p_mutation_without_cross: float = 0.3
    p_interspecies_mate: float = 0.001
    n_min_elite_survival: int = 3       # species needs >= 3 survivors before its champion is copied
    use_random_representatives: bool = True

## Population: speciation and the evolution loop

Two details worth flagging:

**Fitness sharing** — each individual's fitness is divided by the size of its species, so a species
cannot win merely by being large. Offspring are then allocated proportionally to each species' shared
fitness.

**Negative fitness** — LunarLander and BipedalWalker return *negative* rewards for the first several
generations. Proportional allocation on negative sums is meaningless, so fitness is shifted so its
minimum is >= 0 before sharing. (The previous version clamped with `max(0.0001, fitness)`, which
collapsed *every* negative fitness to the same value and removed species-level selection pressure
entirely while rewards were negative.)

In [ ]:
def cluster_species(individuals: list, previous_representatives: dict,
                    compatibility_threshold: float = 3.0) -> list:
    """Greedy sequential clustering against the previous generation's representatives."""
    representatives = previous_representatives.copy()
    clusters = []
    next_cluster_id = max(representatives.keys()) + 1 if representatives else 0

    for ind in individuals:
        assigned = None
        for cluster_id, rep in representatives.items():
            if ind.distance(rep) < compatibility_threshold:
                assigned = cluster_id
                break
        if assigned is None:                       # no species fits -> found a new one
            assigned = next_cluster_id
            representatives[next_cluster_id] = ind
            next_cluster_id += 1
        clusters.append(assigned)
    return clusters


def no_speciation(individuals: list, previous_representatives: dict,
                  compatibility_threshold: float = 3.0) -> list:
    """Speciation switched off: the whole population is one single species (for Analysis 1)."""
    return [0] * len(individuals)


class Population:
    """NEAT speciation + evolution loop (evaluate -> speciate -> select -> reproduce -> mutate)."""

    def __init__(self, size: int, compatibility_threshold: float, cluster_species_func: Callable,
                 config: PopulationConfig = None):
        self.config = config if config else PopulationConfig()
        self.size = size
        self.individuals = []
        self.cluster_species_func = cluster_species_func
        self.population_counter = {'innov': 0, 'node': 0}
        self.current_generation_innovations = {}
        self.species_history = defaultdict(lambda: {'max_fitness': -float('inf'), 'stagnant_gens': 0})
        self.previous_representatives = {}
        self.compatibility_threshold = compatibility_threshold

    # --- innovation bookkeeping: same structural mutation in one generation -> same number ---
    def _get_link_innovation(self, in_node: int, out_node: int) -> int:
        key = ('link', in_node, out_node)
        if key not in self.current_generation_innovations:
            self.population_counter['innov'] += 1
            self.current_generation_innovations[key] = self.population_counter['innov']
        return self.current_generation_innovations[key]

    def _get_node_innovation(self, link_to_split: int) -> int:
        key = ('node', link_to_split)
        if key not in self.current_generation_innovations:
            self.population_counter['node'] += 1
            self.current_generation_innovations[key] = self.population_counter['node']
        return self.current_generation_innovations[key]

    def reset_generation_innovations(self):
        self.current_generation_innovations.clear()

    # --- fitness evaluation: the whole population in parallel, one env per individual ---
    def evaluate_fitness(self, vector_env, action_mapper=None, max_steps=1000, n_runs=3, seed_base=None):
        if vector_env.num_envs != self.size:
            raise ValueError(f"VectorEnv must have exactly {self.size} environments.")

        total_scores = np.zeros(self.size, dtype=float)

        for run in range(n_runs):
            # every individual of a generation faces the SAME episodes -> fair comparison,
            # and the episodes change from generation to generation -> no overfitting to one start
            if seed_base is None:
                obs, _ = vector_env.reset()
            else:
                obs, _ = vector_env.reset(seed=[int(seed_base + run)] * self.size)

            finished = np.zeros(self.size, dtype=bool)
            run_scores = np.zeros(self.size, dtype=float)

            for _ in range(max_steps):
                if finished.all():
                    break
                actions = []
                for i, ind in enumerate(self.individuals):
                    if not finished[i]:
                        raw = ind.forward_pass(obs[i])
                        actions.append(action_mapper(raw) if action_mapper else raw)
                    else:
                        actions.append(vector_env.single_action_space.sample())   # dummy, masked out

                obs, rewards, terminated, truncated, _ = vector_env.step(np.array(actions))
                run_scores += rewards * (~finished)
                finished |= (terminated | truncated)

            total_scores += run_scores

        for ind, fit in zip(self.individuals, total_scores / n_runs):
            ind.fitness = float(fit)

    # --- one generation of selection / reproduction / mutation ---
    def select_reproduce_mutate(self):
        report = {}
        clusters = self.cluster_species_func(self.individuals, self.previous_representatives,
                                             self.compatibility_threshold)
        n_per_cluster = Counter(clusters)

        # FIX: shift fitness to >= 0 instead of clamping negatives to a constant.
        raw = np.array([ind.fitness for ind in self.individuals], dtype=float)
        shifted = raw - raw.min() + 1e-6
        shared = shifted / np.array([n_per_cluster[c] for c in clusters])

        report['species'] = {}
        fitness_per_cluster = {}
        global_max = raw.max()
        for cluster in set(clusters):
            members = [ind for ind, c in zip(self.individuals, clusters) if c == cluster]
            max_fit = max(ind.fitness for ind in members)
            report['species'][cluster] = [ind.fitness for ind in members]

            hist = self.species_history[cluster]
            if max_fit > hist['max_fitness']:
                hist['max_fitness'] = max_fit
                hist['stagnant_gens'] = 0
            else:
                hist['stagnant_gens'] += 1

            is_stagnant = hist['stagnant_gens'] >= self.config.dropoff_age
            is_best_species = (max_fit == global_max)      # the champion's species is never culled

            if is_stagnant and not is_best_species:
                fitness_per_cluster[cluster] = 1e-6
            else:
                fitness_per_cluster[cluster] = float(np.sum(shared[np.array(clusters) == cluster]))

        report['sharedfitness_per_cluster'] = fitness_per_cluster

        total = sum(fitness_per_cluster.values())
        allocation = {c: int((f / total) * self.size) for c, f in fitness_per_cluster.items()}
        missing = self.size - sum(allocation.values())
        if missing > 0:
            allocation[max(fitness_per_cluster, key=fitness_per_cluster.get)] += missing
        report['allocations'] = allocation

        # survivors per species
        survivor_map, all_survivors = {}, []
        for cluster in set(clusters):
            members = sorted([ind for ind, c in zip(self.individuals, clusters) if c == cluster],
                             key=lambda i: i.fitness, reverse=True)
            n_surv = max(1, int(len(members) * self.config.p_selection))
            survivor_map[cluster] = members[:n_surv]
            all_survivors.extend(members[:n_surv])

        next_generation = []
        for cluster, n_offspring in allocation.items():
            if n_offspring == 0:
                continue
            survivors = survivor_map[cluster]
            count = 0

            if len(survivors) >= self.config.n_min_elite_survival:
                next_generation.append(survivors[0].clone())     # elitism: champion unchanged
                count += 1

            while count < n_offspring:
                if np.random.random() < self.config.p_mutation_without_cross or len(survivors) < 2:
                    child = np.random.choice(survivors).clone()
                else:
                    i, j = np.random.choice(len(survivors), 2, replace=False)
                    parent1, parent2 = survivors[i], survivors[j]
                    if np.random.random() < self.config.p_interspecies_mate:
                        parent2 = np.random.choice(all_survivors)
                    child = parent1.crossover(parent2)

                child.mutate_weights()
                child.mutate_genes(self._get_link_innovation, self._get_node_innovation)
                child.compile_network()
                next_generation.append(child)
                count += 1

        # representatives for next generation's clustering
        self.previous_representatives.clear()
        for cluster in set(clusters):
            survivors = survivor_map.get(cluster, [])
            if not survivors:
                continue
            self.previous_representatives[cluster] = (np.random.choice(survivors)
                                                      if self.config.use_random_representatives
                                                      else survivors[0])

        self.individuals = next_generation
        self.reset_generation_innovations()
        return report

## Building the initial population

Every genome starts *minimal*: inputs fully connected to outputs, **no hidden nodes**. Structure has to
earn its place through mutation, which is exactly NEAT's complexification principle.

In [ ]:
def create_population(size: int, n_inputs: int, n_outputs: int, compatibility_threshold: float,
                      cluster_func: Callable, genome_config: GenomeConfig = None,
                      pop_config: PopulationConfig = None) -> Population:
    pop = Population(size=size, compatibility_threshold=compatibility_threshold,
                     cluster_species_func=cluster_func, config=pop_config)
    pop.population_counter['node'] = n_inputs + n_outputs - 1

    for _ in range(size):
        ind = Individual(genome_config)
        for i in range(n_inputs):
            ind.add_node_gene(node_id=i, node_type=NodeType.INPUT)
        for j in range(n_outputs):
            ind.add_node_gene(node_id=n_inputs + j, node_type=NodeType.OUTPUT)

        for in_node in range(n_inputs):
            for j in range(n_outputs):
                out_node = n_inputs + j
                ind.add_connection_gene(
                    in_node=in_node, out_node=out_node,
                    innov=pop._get_link_innovation(in_node, out_node),
                    weight=float(np.random.uniform(ind.config.mut_min_weight,
                                                   ind.config.mut_max_weight)),
                    enabled=True)
        ind.compile_network()
        pop.individuals.append(ind)

    pop.reset_generation_innovations()
    return pop

## Training loop

Per generation: evaluate -> record -> select/reproduce/mutate -> adapt the compatibility threshold.

The threshold is regulated towards a **target species count** rather than being fixed, because a
BipedalWalker genome (24 inputs, 96 initial links) has a very different distance scale from a CartPole
genome (4 inputs, 8 links).

The report now also tracks **best/mean fitness** and **network complexity** per generation, which the
analyses below need.

In [ ]:
def train_NEAT(gym_env_func: Callable, action_mapper: Callable, max_steps: int, n_runs: int,
               max_iterations: int, population_size: int, compatibility_threshold: float,
               n_target_species: int = 0, genome_config: GenomeConfig = None,
               pop_config: PopulationConfig = None, use_speciation: bool = True,
               seed: int = 0, verbose: bool = True):
    np.random.seed(seed)                       # reproducible runs (needed for the multi-seed analyses)

    sample_env = gym_env_func()
    n_inputs = sample_env.observation_space.shape[0]
    n_outputs = (sample_env.action_space.n
                 if isinstance(sample_env.action_space, gym.spaces.Discrete)
                 else sample_env.action_space.shape[0])
    sample_env.close()

    envs = gym.vector.AsyncVectorEnv([lambda: gym_env_func() for _ in range(population_size)])
    population = create_population(
        size=population_size, n_inputs=n_inputs, n_outputs=n_outputs,
        compatibility_threshold=compatibility_threshold,
        cluster_func=cluster_species if use_speciation else no_speciation,
        genome_config=genome_config, pop_config=pop_config)

    best_individual, report = None, []

    for i in range(max_iterations):
        population.evaluate_fitness(envs, action_mapper=action_mapper, max_steps=max_steps,
                                    n_runs=n_runs, seed_base=seed * 100000 + i * 7)
        population.individuals.sort(key=lambda ind: ind.fitness, reverse=True)

        champion = population.individuals[0]
        if best_individual is None or champion.fitness > best_individual.fitness:
            best_individual = champion.clone()          # clone() now preserves fitness

        fits = np.array([ind.fitness for ind in population.individuals])
        gen_report = {
            'generation': i,
            'best_fitness': float(fits.max()),
            'mean_fitness': float(fits.mean()),
            'mean_nodes': float(np.mean([ind.network.number_of_nodes() for ind in population.individuals])),
            'mean_edges': float(np.mean([ind.n_enabled_edges() for ind in population.individuals])),
            'best_genome': champion.clone(),
            'worst_genome': population.individuals[-1].clone(),
        }

        gen_report.update(population.select_reproduce_mutate())
        gen_report['representatives'] = list(population.previous_representatives.values())
        report.append(gen_report)

        n_species = len(gen_report['species'])
        if verbose:
            print(f"Gen [{i:02}] best={gen_report['best_fitness']:8.2f} "
                  f"mean={gen_report['mean_fitness']:8.2f} species={n_species:3d} "
                  f"threshold={population.compatibility_threshold:.3f} "
                  f"nodes={gen_report['mean_nodes']:.1f} edges={gen_report['mean_edges']:.1f}")

        if n_target_species:                    # adaptive compatibility threshold
            if n_species > n_target_species:
                population.compatibility_threshold *= 1.05
            elif n_species < n_target_species:
                population.compatibility_threshold *= 0.95
            population.compatibility_threshold = max(0.5, population.compatibility_threshold)

    envs.close()
    return best_individual, report

## Action mappers and held-out evaluation

`evaluate_heldout` re-tests a finished genome on **fixed seeds that evolution never saw**. This is the
number that actually counts — the fitness reached *during* evolution is the maximum over thousands of
noisy estimates and is therefore optimistically biased.

In [ ]:
def argmax_action_mapper(network_output):
    return int(np.argmax(network_output))


def continuous_action_mapper(network_output):
    # tanh already yields [-1, 1], which is exactly BipedalWalker's action range
    return np.asarray(network_output)


def evaluate_heldout(model, env_name, action_mapper, n_episodes=100, seed_base=7000, max_steps=1600):
    env = gym.make(env_name)
    scores = []
    for i in range(n_episodes):
        state, _ = env.reset(seed=seed_base + i)
        total, done, steps = 0.0, False, 0
        while not done and steps < max_steps:
            state, reward, terminated, truncated, _ = env.step(action_mapper(model.forward_pass(state)))
            total += reward
            done = terminated or truncated
            steps += 1
        scores.append(total)
    env.close()
    return np.array(scores)


def watch_agent(model, render_env, action_mapper, max_steps=1600):
    """Render one episode (needs a display; not executed in this notebook)."""
    state, _ = render_env.reset()
    total, done, steps = 0.0, False, 0
    while not done and steps < max_steps:
        state, reward, terminated, truncated, _ = render_env.step(
            action_mapper(model.forward_pass(state)))
        total += reward            # FIX: used to be `reward += reward`, which printed nonsense
        done = terminated or truncated
        steps += 1
    render_env.close()
    print(f"Achieved {total:.1f} over {steps} steps")
    return total

## Visualisation helpers

Defined **before** they are used, so the notebook runs top-to-bottom.

In [ ]:
sns.set_theme(style="whitegrid")


def plot_training(report, name):
    """Fitness, species count and network complexity over generations."""
    gens = [r['generation'] for r in report]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))

    ax[0].plot(gens, [r['best_fitness'] for r in report], label='best')
    ax[0].plot(gens, [r['mean_fitness'] for r in report], label='mean')
    ax[0].set_title(f'{name} — fitness'); ax[0].set_xlabel('generation'); ax[0].legend()

    ax[1].plot(gens, [len(r['species']) for r in report], color='tab:green')
    ax[1].set_title('number of species'); ax[1].set_xlabel('generation')

    ax[2].plot(gens, [r['mean_nodes'] for r in report], label='nodes')
    ax[2].plot(gens, [r['mean_edges'] for r in report], label='enabled edges')
    ax[2].set_title('network complexity (mean)'); ax[2].set_xlabel('generation'); ax[2].legend()

    plt.tight_layout(); plt.show()


def visualize_species_evolution(report):
    """Per-species fitness trajectories and the population composition over time."""
    n_gen = len(report)
    gens = np.arange(n_gen)
    all_ids = sorted({s for r in report for s in r['species']})
    idx = {s: i for i, s in enumerate(all_ids)}

    mean_fit = np.full((len(all_ids), n_gen), np.nan)
    max_fit = np.full((len(all_ids), n_gen), np.nan)
    sizes = np.zeros((len(all_ids), n_gen))

    for g, r in enumerate(report):
        for s, fits in r['species'].items():
            if fits:
                mean_fit[idx[s], g] = np.mean(fits)
                max_fit[idx[s], g] = np.max(fits)
                sizes[idx[s], g] = len(fits)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
    palette = sns.color_palette("tab20", n_colors=max(len(all_ids), 1))

    for i, s in enumerate(all_ids):
        if np.any(~np.isnan(mean_fit[i])):
            ax1.plot(gens, mean_fit[i], color=palette[i], linewidth=2, alpha=0.8,
                     label=f"species {s}")
            ax1.plot(gens, max_fit[i], color=palette[i], linestyle='--', linewidth=1, alpha=0.5)
    ax1.set_title("species fitness (solid = mean, dashed = max)")
    ax1.set_ylabel("fitness")
    if len(all_ids) <= 20:
        ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize="small",
                   ncol=2 if len(all_ids) > 10 else 1)

    ax2.stackplot(gens, sizes, colors=palette, alpha=0.85)
    ax2.set_title("species sizes (population composition)")
    ax2.set_xlabel("generation"); ax2.set_ylabel("individuals")
    ax2.set_xlim(0, max(1, n_gen - 1))
    plt.tight_layout(); plt.show()


def visualize_network(individual, title="NEAT network", ax=None):
    """Inputs left, outputs right, hidden nodes layered by depth."""
    G = individual.network
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    inputs = [n for n, a in G.nodes(data=True) if a['type'] == NodeType.INPUT]
    outputs = [n for n, a in G.nodes(data=True) if a['type'] == NodeType.OUTPUT]
    hiddens = [n for n, a in G.nodes(data=True) if a['type'] == NodeType.HIDDEN]

    layers = {n: 0 for n in inputs}
    for n in hiddens:
        anc = nx.ancestors(G, n) | {n}
        try:
            layers[n] = max(1, nx.dag_longest_path_length(G.subgraph(anc)))
        except nx.NetworkXError:
            layers[n] = 1
    out_layer = (max(layers.values()) if layers else 0) + 1
    for n in outputs:
        layers[n] = out_layer

    nx.set_node_attributes(G, layers, 'layer')
    pos = nx.multipartite_layout(G, subset_key='layer')

    colors = ['#4a90e2' if G.nodes[n]['type'] == NodeType.INPUT else
              '#e67e22' if G.nodes[n]['type'] == NodeType.OUTPUT else '#2ecc71' for n in G.nodes]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=550,
                           edgecolors='black', linewidths=1.2)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color='white', font_weight='bold')

    enabled = [(u, v, a) for u, v, a in G.edges(data=True) if a['enabled']]
    disabled = [(u, v) for u, v, a in G.edges(data=True) if not a['enabled']]
    max_w = max([abs(a['weight']) for _, _, a in enabled], default=1.0) or 1.0

    for u, v, a in enabled:
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=[(u, v)],
                               width=1.0 + 4.0 * abs(a['weight']) / max_w,
                               edge_color='#e74c3c' if a['weight'] < 0 else '#2980b9',
                               alpha=0.8, arrowsize=12, connectionstyle="arc3,rad=0.05")
    if disabled:
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=disabled, width=1.0, edge_color='#95a5a6',
                               style='dashed', alpha=0.3, arrowsize=8,
                               connectionstyle="arc3,rad=0.05")
    ax.set_title(title, fontweight='bold'); ax.axis('off')
    return ax

# Benchmark 1 — CartPole

4 inputs, 2 discrete actions, solved = 500. The sanity check: if this does not reach 500, the
implementation is broken.

In [ ]:
make_cartpole = lambda: gym.make("CartPole-v1")

best_cartpole, report_cartpole = train_NEAT(
    gym_env_func=make_cartpole, action_mapper=argmax_action_mapper,
    max_steps=500, n_runs=2, max_iterations=15, population_size=100,
    compatibility_threshold=3.0, n_target_species=6, seed=0)

scores = evaluate_heldout(best_cartpole, 'CartPole-v1', argmax_action_mapper, max_steps=500)
print(f"\nCartPole held-out over 100 unseen episodes: {scores.mean():.1f} "
      f"(min {scores.min():.0f} / max {scores.max():.0f} / std {scores.std():.1f})")
print(f"Best genome: {best_cartpole.network.number_of_nodes()} nodes "
      f"({best_cartpole.n_hidden()} hidden), {best_cartpole.n_enabled_edges()} enabled edges")

plot_training(report_cartpole, 'CartPole')
visualize_network(best_cartpole, "CartPole — best genome"); plt.show()

# Benchmark 2 — LunarLander

8 inputs, 4 discrete actions, solved = 200. Rewards start out **negative** (every lander crashes at
first), which is exactly the regime in which the offspring allocation must handle negative fitness
correctly.

In [ ]:
make_lunar = lambda: gym.make('LunarLander-v3')

best_lunar, report_lunar = train_NEAT(
    gym_env_func=make_lunar, action_mapper=argmax_action_mapper,
    max_steps=1000, n_runs=3, max_iterations=50, population_size=150,
    compatibility_threshold=1.8, n_target_species=10, seed=0)

scores_lunar = evaluate_heldout(best_lunar, 'LunarLander-v3', argmax_action_mapper, max_steps=1000)
print(f"\nLunarLander held-out over 100 unseen episodes: {scores_lunar.mean():.1f} (solved >= 200)")
print(f"Best genome: {best_lunar.network.number_of_nodes()} nodes "
      f"({best_lunar.n_hidden()} hidden), {best_lunar.n_enabled_edges()} enabled edges")

plot_training(report_lunar, 'LunarLander')
visualize_species_evolution(report_lunar)

### Successful vs. unsuccessful network

The best and the worst genome of the final generation, side by side.

In [ ]:
best_g = report_lunar[-1]['best_genome']
worst_g = report_lunar[-1]['worst_genome']
s_best = evaluate_heldout(best_g, 'LunarLander-v3', argmax_action_mapper, n_episodes=20, max_steps=1000)
s_worst = evaluate_heldout(worst_g, 'LunarLander-v3', argmax_action_mapper, n_episodes=20, max_steps=1000)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
visualize_network(best_g, f"successful (mean {s_best.mean():.0f})", ax=axes[0])
visualize_network(worst_g, f"unsuccessful (mean {s_worst.mean():.0f})", ax=axes[1])
plt.tight_layout(); plt.show()

# Benchmark 3 — BipedalWalker

24 inputs, **4 continuous** actions in [-1, 1]. This is why the activation is `tanh` and the action
mapper is the identity — there is no `argmax` here. Solved would be ~300; we report honest progress.

In [ ]:
make_bipedal = lambda: gym.make('BipedalWalker-v3')

best_bipedal, report_bipedal = train_NEAT(
    gym_env_func=make_bipedal, action_mapper=continuous_action_mapper,
    max_steps=800, n_runs=1, max_iterations=40, population_size=100,
    compatibility_threshold=2.1, n_target_species=10, seed=0)

scores_bw = evaluate_heldout(best_bipedal, 'BipedalWalker-v3', continuous_action_mapper,
                             n_episodes=20, max_steps=800)
print(f"\nBipedalWalker held-out over 20 unseen episodes: {scores_bw.mean():.1f} (solved ~300)")
print(f"Best genome: {best_bipedal.network.number_of_nodes()} nodes "
      f"({best_bipedal.n_hidden()} hidden), {best_bipedal.n_enabled_edges()} enabled edges")

plot_training(report_bipedal, 'BipedalWalker')

# Analysis 1 — How does speciation affect performance?

`use_speciation=False` puts the whole population into a single species: fitness sharing and the
protected niches disappear, and every genome competes directly against every other. Without that
protection a newly mutated topology — which is always worse at first, because its new weights are
untuned — is eliminated before it can mature.

Identical configuration, identical seeds, one single difference. Averaged over 2 seeds.

In [ ]:
def mean_best_curve(use_speciation, seeds=(0, 1), **kw):
    curves = []
    for s in seeds:
        _, rep = train_NEAT(use_speciation=use_speciation, seed=s, verbose=False, **kw)
        curves.append([r['best_fitness'] for r in rep])
    return np.mean(np.array(curves), axis=0)


base = dict(gym_env_func=make_lunar, action_mapper=argmax_action_mapper, max_steps=1000,
            n_runs=2, max_iterations=15, population_size=60, compatibility_threshold=1.8,
            n_target_species=8)

with_spec = mean_best_curve(True, **base)
without_spec = mean_best_curve(False, **base)

plt.figure(figsize=(9, 5))
plt.plot(with_spec, label='with speciation')
plt.plot(without_spec, label='without speciation')
plt.xlabel('generation'); plt.ylabel('best fitness (mean of 2 seeds)')
plt.title('LunarLander — effect of speciation'); plt.legend(); plt.show()

# Analysis 2 — Which mutation rates give stable results?

Sweep of `mut_p_add_node`. Too low and the network can never build the structure the task needs; too
high and the networks bloat with untuned weights faster than evolution can optimise them.

Error bars are the standard deviation over 2 seeds.

In [ ]:
rates = [0.0, 0.02, 0.05, 0.15, 0.3]
means, stds = [], []

for r in rates:
    finals = []
    for s in [0, 1]:
        _, rep = train_NEAT(
            gym_env_func=make_lunar, action_mapper=argmax_action_mapper, max_steps=1000,
            n_runs=2, max_iterations=12, population_size=60, compatibility_threshold=1.8,
            n_target_species=8, seed=s, verbose=False,
            genome_config=GenomeConfig(mut_p_add_node=r))       # only this one value changes
        finals.append(rep[-1]['best_fitness'])
    means.append(np.mean(finals)); stds.append(np.std(finals))
    print(f"mut_p_add_node={r:<5} -> best fitness {np.mean(finals):8.2f} (+/- {np.std(finals):.2f})")

plt.figure(figsize=(8, 5))
plt.errorbar([str(r) for r in rates], means, yerr=stds, marker='o', capsize=4)
plt.xlabel('mut_p_add_node'); plt.ylabel('best fitness after 12 generations (mean of 2 seeds)')
plt.title('LunarLander — effect of the add-node mutation rate'); plt.show()

# Analysis 3 — How does network complexity evolve?

NEAT starts minimal (no hidden nodes, inputs wired straight to outputs) and only grows when growth pays
for itself. The curves come from the three benchmark runs above — nothing is retrained.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for rep, name in [(report_cartpole, 'CartPole'), (report_lunar, 'LunarLander'),
                  (report_bipedal, 'BipedalWalker')]:
    ax[0].plot([r['mean_nodes'] for r in rep], label=name)
    ax[1].plot([r['mean_edges'] for r in rep], label=name)
ax[0].set_title('mean number of nodes'); ax[0].set_xlabel('generation'); ax[0].legend()
ax[1].set_title('mean number of enabled edges'); ax[1].set_xlabel('generation'); ax[1].legend()
plt.tight_layout(); plt.show()

for model, name, n_in, n_out in [(best_cartpole, 'CartPole', 4, 2),
                                 (best_lunar, 'LunarLander', 8, 4),
                                 (best_bipedal, 'BipedalWalker', 24, 4)]:
    print(f"{name:15s} minimal start: {n_in + n_out:2d} nodes / {n_in * n_out:3d} edges "
          f"-> best genome: {model.network.number_of_nodes():2d} nodes / "
          f"{model.n_enabled_edges():3d} edges  ({model.n_hidden()} hidden nodes evolved)")

# Genetic distance between species

Distance matrix over the species representatives of the final LunarLander generation. The diagonal is 0
by definition; bright off-diagonal cells mean two species really are genetically far apart — evidence
that speciation formed genuinely separate niches rather than arbitrary labels.

In [ ]:
reps = report_lunar[-1]['representatives']
D = np.array([[a.distance(b) for b in reps] for a in reps])

plt.figure(figsize=(7, 6))
sns.heatmap(D, cmap='viridis', square=True, cbar_kws={'label': 'compatibility distance'})
plt.title(f'genetic distance between {len(reps)} species representatives (LunarLander)')
plt.xlabel('species'); plt.ylabel('species'); plt.show()

# Conclusion

- **CartPole** is solved reliably and needs almost no hidden structure — NEAT keeps the network close to
  the minimal starting topology.
- **LunarLander** learns a usable policy; the reported number is a **held-out** average over 100 unseen
  episodes, not the (optimistically biased) fitness reached during evolution.
- **BipedalWalker** shows clear progress but is not solved; that would need considerably more compute.
- **Speciation** measurably stabilises the search by protecting new topologies (Analysis 1).
- **Mutation rates** have a clear optimum: 0 prevents the structure the task needs, very high rates bloat
  the networks faster than evolution can tune them (Analysis 2).
- **Complexity** grows only as far as the task demands (Analysis 3), which is the core claim of NEAT.